In [5]:
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.6/28.6 MB 151.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 124.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 52.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 65.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 49.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 41.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.9 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [unsloth

In [6]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

# 1. Load the small 3B parameter model in 4-bit mode
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 2. Add the "Sticky Notes" (LoRA adapters) for fast training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)
print("✅ Small Robot Loaded and Ready!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.10.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
✅ Small Robot Loaded and Ready!


In [7]:

import pandas as pd
from datasets import Dataset
from huggingface_hub import hf_hub_download, list_repo_files

repo_id = "Enaksheexe/enakshee_tech_stack_dataset"


files = list_repo_files(repo_id, repo_type="dataset")
csv_file = [f for f in files if f.endswith(".csv")][0]
print(f"Found dataset file: {csv_file}")


file_path = hf_hub_download(repo_id=repo_id, filename=csv_file, repo_type="dataset")

with open(file_path, "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

header_row_idx = 0
for idx, line in enumerate(lines):
    if "Flashcard ID" in line and "instruction" in line:
        header_row_idx = idx
        break

# 3. Load into Pandas starting from the actual column names
df = pd.read_csv(file_path, skiprows=header_row_idx)
df.columns = df.columns.str.strip()  # Clean any accidental spaces

# Drop completely empty rows
df = df.dropna(
    subset=[
        "instruction (System Role)",
        "input (User Project Constraints)",
        "output (Target LLM Recommendation)",
    ]
)

# Convert to a Hugging Face Dataset object
dataset = Dataset.from_pandas(df, preserve_index=False)

# 4. Format into the Alpaca prompt structure for Unsloth
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Make sure EOS_TOKEN is defined from Block 2 (tokenizer.eos_token)
EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples):
    instructions = examples["instruction (System Role)"]
    inputs = examples["input (User Project Constraints)"]
    outputs = examples["output (Target LLM Recommendation)"]
    texts = []
    for instruction, input_text, output_text in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output_text) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}


# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"\nSuccessfully loaded {len(dataset)} flashcards!")
print("\n--- SAMPLE FORMATTED PROMPT ---")
print(dataset["text"][0])

Found dataset file: TiesVerseProject_Enakshee_Mondal.xlsx - LLM_Training_Flashcards (1).csv


Map:   0%|          | 0/40 [00:00<?, ? examples/s]


Successfully loaded 40 flashcards!

--- SAMPLE FORMATTED PROMPT ---
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert Software Architect AI. Analyze the user's project details, team size, timeline, budget, developer experience, and expected scale, then recommend the ideal production-ready tech stack with clear industry reasoning.

### Input:
Project Category: SaaS / Web App | Specific Use Case: B2B Micro-SaaS MVP (Invoicing & Subscription tool) | Team Size: Solo (1 Person) | Timeline: 2 Weeks | Budget: Free / $0 | Team Experience: Beginner | Expected Scale: Low (<1k users/mo)

### Response:
Recommended Stack -> Frontend: Next.js (React) + Tailwind CSS | Backend: Next.js Server Actions / API Routes | Database & Storage: Supabase (PostgreSQL) | Auth & Billing: Supabase Auth + Stripe | Hosting: Vercel (Free Tier) || Why This Stack: Single codebas

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer.train()
print("TRAINING COMPLETE!")

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/40 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40 | Num Epochs = 12 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss
5,3.315947
10,2.462360
15,1.746791
20,1.276804
25,1.069728
30,0.924448
35,0.828135
40,0.758544
45,0.700121
50,0.649213


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


TRAINING COMPLETE!


In [9]:
FastLanguageModel.for_inference(model)

test_prompt = alpaca_prompt.format(
    "You are an expert Software Architect AI. Analyze the user's project details, team size, timeline, budget, developer experience, and expected scale, then recommend the ideal production-ready tech stack with clear industry reasoning.",
    "Project Category: Mobile App | Specific Use Case: Campus Food Delivery App | Team Size: Solo (1 Person) | Timeline: 3 Weeks | Budget: Free / $0 | Team Experience: Beginner | Expected Scale: Low (<2k users)",
    "" # Leave output blank so the AI fills it in!
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert Software Architect AI. Analyze the user's project details, team size, timeline, budget, developer experience, and expected scale, then recommend the ideal production-ready tech stack with clear industry reasoning.

### Input:
Project Category: Mobile App | Specific Use Case: Campus Food Delivery App | Team Size: Solo (1 Person) | Timeline: 3 Weeks | Budget: Free / $0 | Team Experience: Beginner | Expected Scale: Low (<2k users)

### Response:
Recommended Stack -> Frontend: Flutter (Dart) + Bloc | Backend: Firebase Realtime Database & Cloud Functions | Database & Storage: Firebase Firestore & Cloud Storage | Auth & Billing: Firebase Auth + Custom Merchant Credentials | Hosting: Firebase Cloud Run or Firebase Cloud Functions || Why This Stack: Flutter builds native Android and iOS apps without wri

In [10]:
model.push_to_hub("Enaksheexe/stackverse-llm", token = "")
tokenizer.push_to_hub("Enaksheexe/stackverse-llm", token = "")
print(" Your custom AI is now live on Hugging Face at Enaksheexe/stackverse-llm!")

README.md:   0%|          | 0.00/573 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  559kB /  120MB            

Validating                    :           |   0%            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp84p6ezin/tokenizer.json:   0%|          | 27.9kB / 11.4MB            

Validating                    :           |   0%            

 Your custom AI is now live on Hugging Face at Enaksheexe/stackverse-llm!


In [11]:
# Switch model to fast inference mode
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [
        alpaca_prompt.format(
            "You are an expert software architect. Recommend an optimal tech stack based on the project constraints.",  # instruction
            "Project: Real-time collaborative code editor for college students. Budget: Free tier only. Team skills: JavaScript/React. Must support 200 concurrent users with low latency.",  # input
            "",  #
        )
    ],
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=256)

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert software architect. Recommend an optimal tech stack based on the project constraints.

### Input:
Project: Real-time collaborative code editor for college students. Budget: Free tier only. Team skills: JavaScript/React. Must support 200 concurrent users with low latency.

### Response:


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Recommended Stack -> Frontend: Next.js + Live Editor UI (CodeMirror or Monaco) | Backend: Go (Golang) + Golang NetRPC (or Rocketeer) | Database & Storage: PostgreSQL + Redis | Auth & Billing: GitHub OAuth / JWT | Hosting: Render (Go) + Vercel (JavaScript) || Why This Stack: Go handles thousands of concurrent WebSocket connections on the server side without burning through $100/m EC2 instances.<|im_end|>
